<a href="https://colab.research.google.com/github/tutonat/Nati_Subtitles_to_Speech/blob/main/Nati_Subtitles_to_Speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NATI TOOLS · Generar audio largo en Google Colab (GPU gratis)

Programa para Google Colab que lee un `.srt` (o un texto) con **tu voz clonada** de NATI TOOLS,
usando la GPU T4 gratuita de Colab. Hace lo mismo que el Panel 2 (mismo motor XTTS, mismos ajustes),
pero mucho más rápido: un texto de 2000 caracteres tarda menos de un minuto.

Necesitas dos archivos de NATI TOOLS:
- **El vector de tu voz** (`vector_xtts_speaker.pt`): botón «Descargar vector» del Panel 2.
- **Tu `.srt`** (o el texto): el que genera el Panel 1, o el tuyo.


## Cómo usarlo

1. En la derecha hay una flecha para que actives o desactives **"visibilidad del encabezado"** para que aparezca el otro menú y puedas cambiar el **entorno de ejecución.**
2. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
3. Crea **3 celdas de código** y pega en cada una el bloque correspondiente de abajo.
4. Ejecuta las celdas **en orden** (botón ▶ de cada una). La celda 1 tarda unos minutos la primera vez.
   Si Colab pide **reiniciar la sesión** al terminar la celda 1, acepta y sigue con la celda 2.
5. Al terminar la celda 3 aparece un enlace `https://….gradio.live`: ábrelo, sube tu vector y tu
   `.srt`, elige el idioma y pulsa **Generar audio**.


## **Celda 1 · Instalación**
# Motor XTTS y la interfaz. Tarda unos minutos la primera vez.

In [ ]:
#@title 1 · Instalar (pulsa ▶ y espera unos minutos) { display-mode: "form" }
!pip install -q "coqui-tts==0.27.5" "transformers==4.57.6" gradio librosa soundfile pypinyin cutlet "fugashi[unidic-lite]"

## **Celda 2 · Motor de lectura (igual que el Panel 2 de NATI TOOLS)**

In [ ]:
#@title 2 · Cargar el modelo de voz (pulsa ▶; solo una vez por sesión) { display-mode: "form" }
import os, re, tempfile
import numpy as np
import soundfile as sf
import librosa
import torch
from huggingface_hub import hf_hub_download

os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 24000

# --- Ajustes de lectura: los mismos que el Panel 2 ---
READ_SETTINGS = {"temperature": 0.03, "top_k": 5, "top_p": 0.80, "repetition_penalty": 4.5, "length_penalty": 1.0}
QUESTION_TEMPERATURE = 0.2           # las frases con pregunta, algo más libres
RETRY_TEMPERATURES = (0.3, 0.6)      # si una frase sale demasiado larga (balbuceo), se repite
SPEED = 1.0                          # velocidad normal
SENTENCE_GAP = 0.15                  # pausa entre frases al leer texto seguido, sin tiempos (s)
MAX_PAUSE = 0.8                      # silencio máximo dentro de una frase (s)
# Doblaje con un .srt: cada frase en su tiempo del video; si no cabe en su hueco, más rápida (tope).
DUB_MAX_SPEED = 1.35
DUB_MARGIN = 0.1
DUB_GROUP_CHARS = 120
PIECE_CHARS = 200
OUTPUT_PEAK_DB = -1.0
MIN_PIECE_CHARS = {"ko": 30, "zh-cn": 12, "ja": 15}
DEFAULT_MIN_PIECE_CHARS = 20
SECONDS_PER_CHAR = {"zh-cn": 0.2, "ko": 0.17, "ja": 0.12, "hi": 0.13}
DEFAULT_SECONDS_PER_CHAR = 0.1
MAX_LENGTH_RATIO = 1.7
QUESTION_MARKS = ("?", "？", "؟")
LANGUAGES = {"Español": "es", "English": "en", "French": "fr", "German": "de", "Italian": "it",
             "Portuguese": "pt", "Polish": "pl", "Turkish": "tr", "Russian": "ru", "Dutch": "nl",
             "Chinese": "zh-cn", "Japanese": "ja", "Hungarian": "hu", "Korean": "ko", "Hindi": "hi"}

# --- Modelo XTTS-v2 (~1,8 GB, se descarga una vez por sesión) ---
MODEL_DIR = "/content/xtts" if os.path.isdir("/content") else os.path.join(tempfile.gettempdir(), "xtts")
for name in ("config.json", "vocab.json", "model.pth", "speakers_xtts.pth"):
    if not os.path.exists(os.path.join(MODEL_DIR, name)):
        # token=False: el modelo es público; así Colab no pide acceso al secreto HF_TOKEN.
        hf_hub_download("coqui/XTTS-v2", name, local_dir=MODEL_DIR, token=False)
config = XttsConfig()
config.load_json(os.path.join(MODEL_DIR, "config.json"))
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=MODEL_DIR, eval=True)
model.to(DEVICE)
print("XTTS listo en", DEVICE)


# --- Texto y subtítulos ---
def parse_srt(content):
    """[(texto, inicio_s)] de un .srt."""
    items = []
    for block in re.split(r"\n\s*\n", content.replace("\r", "").strip()):
        lines = [l for l in block.split("\n") if l.strip()]
        times = next((l for l in lines if "-->" in l), None)
        if not times:
            continue
        h, m, s, ms = re.match(r"\s*(\d+):(\d+):(\d+)[,.](\d+)", times).groups()
        text = " ".join(l.strip() for l in lines[lines.index(times) + 1:])
        text = re.sub(r"<[^>]+>", "", text).strip()
        if text:
            items.append((text, int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000))
    return items


def parse_srt_cues(content):
    """[(inicio_s, fin_s, texto)] de un .srt (para el doblaje)."""
    def seconds(h, m, s, ms):
        return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000
    cues = []
    for block in re.split(r"\n\s*\n", content.replace("\r", "").strip()):
        lines = [l for l in block.split("\n") if l.strip()]
        times = next((l for l in lines if "-->" in l), None)
        if not times:
            continue
        found = re.findall(r"(\d+):(\d+):(\d+)[,.](\d+)", times)
        if len(found) < 2:
            continue
        text = re.sub(r"<[^>]+>", "", " ".join(l.strip() for l in lines[lines.index(times) + 1:])).strip()
        if text:
            cues.append((seconds(*found[0]), seconds(*found[1]), text))
    return cues


def split_sentences(text):
    text = " ".join(text.split())
    return [s.strip() for s in re.split(r"(?<=[.!?¡¿。！？…])\s+", text) if s.strip()]


def split_parts(text, max_chars):
    """Frases completas juntas hasta max_chars (sin cortar palabras)."""
    parts, current = [], ""
    for sentence in split_sentences(text):
        while len(sentence) > max_chars:
            cut = sentence.rfind(" ", 0, max_chars)
            cut = cut if cut > 0 else max_chars
            if current:
                parts.append(current); current = ""
            parts.append(sentence[:cut].strip()); sentence = sentence[cut:].strip()
        candidate = f"{current} {sentence}".strip()
        if current and len(candidate) > max_chars:
            parts.append(current); current = sentence
        else:
            current = candidate
    if current:
        parts.append(current)
    return parts


def merge_short(items, language):
    """Las frases muy cortas se leen junto a la siguiente (con poco texto XTTS alucina)."""
    minimum = MIN_PIECE_CHARS.get(language, DEFAULT_MIN_PIECE_CHARS)
    joiner = "" if language in ("zh-cn", "ja") else " "
    merged = []
    for text, start in items:
        if merged and len(merged[-1][0].replace(" ", "")) < minimum:
            merged[-1] = (merged[-1][0] + joiner + text, merged[-1][1])
        else:
            merged.append((text, start))
    if len(merged) > 1 and len(merged[-1][0].replace(" ", "")) < minimum:
        last = merged.pop()
        merged[-1] = (merged[-1][0] + joiner + last[0], merged[-1][1])
    return merged


def format_ts(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


# --- Lectura ---
def shorten_pauses(wav, max_pause=MAX_PAUSE):
    voiced = librosa.effects.split(wav, top_db=35, frame_length=1024, hop_length=256)
    if len(voiced) < 2:
        return wav
    keep, fade = int(max_pause * SAMPLE_RATE), int(0.01 * SAMPLE_RATE)
    pieces, cursor = [], 0
    for (_, end), (start, _) in zip(voiced[:-1], voiced[1:]):
        if start - end > keep:
            piece = wav[cursor:end + keep // 2].copy()
            piece[-fade:] *= np.linspace(1, 0, fade, dtype=np.float32)
            pieces.append(piece)
            cursor = start - keep // 2
    tail = wav[cursor:].copy()
    if pieces:
        tail[:fade] *= np.linspace(0, 1, fade, dtype=np.float32)
    return np.concatenate(pieces + [tail])


# Enganches: a veces XTTS se queda repitiendo el mismo sonido ("aaaa…" con pitido). Se reconoce
# porque el volumen casi no varía y el sonido casi no cambia durante medio segundo o más.
def stuck_runs(wav, window=0.5, max_cv=0.06, max_change=3.5, min_seconds=0.6):
    hop = 256
    if len(wav) < SAMPLE_RATE * window:
        return []
    rms = librosa.feature.rms(y=wav, frame_length=1024, hop_length=hop)[0]
    mfcc = librosa.feature.mfcc(y=wav, sr=SAMPLE_RATE, n_mfcc=13, hop_length=hop, n_fft=1024)[1:]
    change = np.r_[0, np.linalg.norm(np.diff(mfcc, axis=1), axis=0)]
    w = int(window * SAMPLE_RATE / hop)
    loud = rms > 0.1 * rms.max()
    bad = np.zeros(len(rms), dtype=bool)
    for i in range(0, len(rms) - w, 4):
        level = rms[i:i + w]
        if loud[i:i + w].mean() >= 0.95 and level.std() / level.mean() < max_cv and change[i:i + w].mean() < max_change:
            bad[i:i + w] = True
    runs, start = [], None
    for k, flag in enumerate(np.r_[bad, False]):
        if flag and start is None:
            start = k
        elif not flag and start is not None:
            if (k - start) * hop >= min_seconds * SAMPLE_RATE:
                runs.append((start * hop, min(len(wav), k * hop)))
            start = None
    return runs


def cut_runs(wav, runs):
    """Quita los tramos enganchados (último recurso), con fundidos de 10 ms."""
    fade = int(0.01 * SAMPLE_RATE)
    pieces, cursor = [], 0
    for start, end in runs:
        piece = wav[cursor:start].copy()
        if len(piece) > fade:
            piece[-fade:] *= np.linspace(1, 0, fade, dtype=np.float32)
        pieces.append(piece)
        cursor = end
    tail = wav[cursor:].copy()
    if len(tail) > fade:
        tail[:fade] *= np.linspace(0, 1, fade, dtype=np.float32)
    return np.concatenate(pieces + [tail])


def read(vector, text, language, speed=SPEED):
    max_chars = min(PIECE_CHARS, model.tokenizer.char_limits.get(language.split("-")[0], PIECE_CHARS))
    audio = []
    for piece in split_parts(text, max_chars):
        if language == "pt":   # en portugués XTTS a veces lee el punto final ("ponto")
            piece = piece.rstrip(" .") or piece
        limit = (MAX_LENGTH_RATIO * len(piece.replace(" ", "")) *
                 SECONDS_PER_CHAR.get(language, DEFAULT_SECONDS_PER_CHAR) + 0.8) / min(speed, 1.0)
        first = QUESTION_TEMPERATURE if any(m in piece for m in QUESTION_MARKS) else READ_SETTINGS["temperature"]
        retries = tuple(t for t in RETRY_TEMPERATURES if t != first)
        best, best_key, best_stuck = None, None, []
        for temperature in (first,) + retries:
            with torch.no_grad():
                wav = model.inference(piece, language, vector["gpt_cond_latent"], vector["speaker_embedding"],
                                      speed=speed, **dict(READ_SETTINGS, temperature=temperature))["wav"]
            wav, _ = librosa.effects.trim(np.asarray(wav, dtype=np.float32), top_db=35)
            wav = shorten_pauses(wav)
            stuck = stuck_runs(wav)
            key = (bool(stuck), len(wav))   # mejor: sin enganches y, entre esos, el más corto
            if best_key is None or key < best_key:
                best, best_key, best_stuck = wav, key, stuck
            if not stuck and len(wav) / SAMPLE_RATE <= limit:
                break
            print(f"  (se repite un trozo: {'enganche' if stuck else 'demasiado largo'})", flush=True)
        if best_stuck:   # todos los intentos con enganche: se recorta para que no llegue el pitido
            best = cut_runs(best, best_stuck)
        audio.append(best)
    return np.concatenate(audio) if audio else np.zeros(0, dtype=np.float32)


def load_vector(path):
    vector = torch.load(path, map_location=DEVICE, weights_only=True)
    if "gpt_cond_latent" not in vector or "speaker_embedding" not in vector:
        raise ValueError("Ese archivo no es un vector de voz de NATI TOOLS (vector_xtts_speaker.pt).")
    return {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in vector.items()}


def minutes(seconds):
    m, s = divmod(int(seconds), 60)
    return f"{m} min {s:02d} s" if m else f"{s} s"


def speak(vector, items, language, progress=None):
    """items: [(texto, inicio_deseado o None)]. Devuelve (audio, [(inicio, fin, texto)])."""
    import time
    items = merge_short(items, language)
    total_chars = sum(len(t) for t, _ in items) or 1
    pieces, cues, cursor, done_chars, started = [], [], 0.0, 0, time.time()
    print(f"Leyendo {len(items)} frases ({total_chars} caracteres) en {DEVICE}…", flush=True)
    for n, (text, wanted) in enumerate(items, 1):
        wav = read(vector, text, language)
        start = cursor if wanted is None else max(cursor, wanted)
        duration = len(wav) / SAMPLE_RATE
        pieces.append((start, wav))
        cues.append((start, start + duration, text))
        cursor = start + duration + SENTENCE_GAP
        # Avance por caracteres leídos y tiempo restante según la velocidad real de esta lectura.
        done_chars += len(text)
        fraction = done_chars / total_chars
        elapsed = time.time() - started
        remaining = elapsed * (1 - fraction) / fraction
        status = f"{fraction:.0%} · frase {n} de {len(items)} · llevamos {minutes(elapsed)} · quedan ≈ {minutes(remaining)}"
        print(status, flush=True)
        if progress:
            progress(fraction, desc=status)
    audio = np.zeros(int(round(cursor * SAMPLE_RATE)), dtype=np.float32)
    for start, wav in pieces:
        at = int(round(start * SAMPLE_RATE))
        audio[at:at + len(wav)] = wav[:len(audio) - at]
    peak = float(np.abs(audio).max())
    if peak > 0:
        audio *= 10 ** (OUTPUT_PEAK_DB / 20) / peak
    return audio, cues


SENTENCE_END = (".", "!", "?", "…", "。", "！", "？")


def group_cues(cues, language, max_chars=DUB_GROUP_CHARS, max_gap=1.0):
    """Subtítulos [(inicio, fin, texto)] -> frases [(inicio, fin_hueco, texto)] para doblar.
    Los subtítulos automáticos de YouTube son trozos que se solapan: se juntan en frases."""
    joiner = "" if language in ("zh-cn", "ja") else " "
    groups, current, last_end = [], None, None
    for start, end, text in sorted(cues):
        text = " ".join(text.split())
        if not text:
            continue
        if current and (current[2].endswith(SENTENCE_END) or len(current[2]) + len(text) > max_chars
                        or start - last_end > max_gap):
            groups.append(current); current = None
        if current is None:
            current = [start, end, text]
        else:
            current[1] = max(current[1], end); current[2] = current[2] + joiner + text
        last_end = end if last_end is None else max(last_end, end)
    if current:
        groups.append(current)
    return [(g[0], groups[i + 1][0] if i + 1 < len(groups) else g[1], g[2]) for i, g in enumerate(groups)]


def dub(vector, cues, language, progress=None):
    """Doblaje: cada frase en su tiempo del video; si no cabe en su hueco, se lee más rápida
    (hasta DUB_MAX_SPEED). El audio dura como mínimo hasta el final del último subtítulo."""
    import time
    groups = group_cues(cues, language)
    total_chars = sum(len(t) for _, _, t in groups) or 1
    pieces, out_cues, cursor, done_chars, started = [], [], 0.0, 0, time.time()
    print(f"Doblando {len(groups)} frases ({total_chars} caracteres) en {DEVICE}…", flush=True)
    for n, (start, slot_end, text) in enumerate(groups, 1):
        at = max(start, cursor)
        room = slot_end - at - DUB_MARGIN
        wav = read(vector, text, language)
        if room > 0 and len(wav) / SAMPLE_RATE > room:   # no cabe en su hueco: más rápida
            speed = min(DUB_MAX_SPEED, SPEED * (len(wav) / SAMPLE_RATE) / room)
            quicker = read(vector, text, language, speed=speed)
            if len(quicker) < len(wav):
                wav = quicker
        duration = len(wav) / SAMPLE_RATE
        pieces.append((at, wav))
        out_cues.append((at, at + duration, text))
        cursor = at + duration + 0.05
        done_chars += len(text)
        fraction = done_chars / total_chars
        elapsed = time.time() - started
        status = (f"{fraction:.0%} · frase {n} de {len(groups)} · llevamos {minutes(elapsed)} · "
                  f"quedan ≈ {minutes(elapsed * (1 - fraction) / fraction)}")
        print(status, flush=True)
        if progress:
            progress(fraction, desc=status)
    length = max(cursor, groups[-1][1] if groups else 0.0)
    audio = np.zeros(int(round(length * SAMPLE_RATE)), dtype=np.float32)
    for at, wav in pieces:
        i = int(round(at * SAMPLE_RATE))
        audio[i:i + len(wav)] = wav[:len(audio) - i]
    peak = float(np.abs(audio).max())
    if peak > 0:
        audio *= 10 ** (OUTPUT_PEAK_DB / 20) / peak
    return audio, out_cues


print("Motor de lectura listo.")

# **## Celda 3 · Interfaz (abre el enlace que aparece)**

In [ ]:
#@title 3 · Abrir la interfaz (pulsa ▶ y abre el enlace que aparece) { display-mode: "form" }
import gradio as gr


def generate(vector_file, srt_file, text, language_name, progress=gr.Progress()):
    if vector_file is None:
        raise gr.Error("Sube el vector de tu voz (vector_xtts_speaker.pt).")
    language = LANGUAGES[language_name]
    vector = load_vector(vector_file if isinstance(vector_file, str) else vector_file.name)
    if srt_file is not None:
        path = srt_file if isinstance(srt_file, str) else srt_file.name
        with open(path, encoding="utf-8", errors="ignore") as fh:
            content = fh.read()
        srt_cues = parse_srt_cues(content) if "-->" in content else []
        items = [] if srt_cues else [(s, None) for s in split_sentences(content)]
    else:
        srt_cues, items = [], [(s, None) for s in split_sentences(text or "")]
    if not srt_cues and not items:
        raise gr.Error("No hay texto: sube un .srt / .txt o escribe el texto.")
    if srt_cues:   # .srt: doblaje, cada frase en su tiempo del video y misma duración que el video
        audio, cues = dub(vector, srt_cues, language, progress)
    else:          # texto sin tiempos: lectura seguida
        audio, cues = speak(vector, items, language, progress)
    out_dir = tempfile.mkdtemp()
    wav_path = os.path.join(out_dir, f"audio_{language}.wav")
    srt_path = os.path.join(out_dir, f"subtitulos_{language}.srt")
    sf.write(wav_path, audio, SAMPLE_RATE, subtype="PCM_16")
    with open(srt_path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(f"{i}\n{format_ts(a)} --> {format_ts(b)}\n{t}\n" for i, (a, b, t) in enumerate(cues, 1)))
    return wav_path, wav_path, srt_path


with gr.Blocks(title="NATI TOOLS · Audio largo con tu voz") as demo:
    gr.Markdown("## NATI TOOLS · Audio largo con tu voz clonada (GPU de Google Colab)\n"
                "Sube el **vector de tu voz** y tu **.srt** (o escribe el texto). El texto ya debe estar en el "
                "idioma que elijas: aquí no se traduce.\n\n"
                + ("✅ **Usando la GPU** (rápido)." if DEVICE == "cuda" else
                   "⚠️ **Sin GPU: irá muy lento.** En Colab: Entorno de ejecución → Cambiar tipo de entorno "
                   "de ejecución → T4 GPU, y vuelve a ejecutar las celdas.") +
                "\n\nEl avance (porcentaje y tiempo restante) se ve aquí y en la salida de la celda.")
    with gr.Row():
        vector_in = gr.File(label="Vector de tu voz (vector_xtts_speaker.pt)", file_types=[".pt"])
        srt_in = gr.File(label="Subtítulos .srt o texto .txt (opcional si escribes el texto)", file_types=[".srt", ".txt"])
    text_in = gr.Textbox(label="…o escribe el texto aquí", lines=6)
    language_in = gr.Dropdown(list(LANGUAGES), value="Español", label="Idioma del texto")
    button = gr.Button("Generar audio", variant="primary")
    audio_out = gr.Audio(label="Resultado", type="filepath")
    with gr.Row():
        wav_out = gr.File(label="Descargar audio (.wav)")
        srt_out = gr.File(label="Descargar subtítulos (.srt) con los tiempos del audio")
    button.click(generate, [vector_in, srt_in, text_in, language_in], [audio_out, wav_out, srt_out])

demo.queue().launch(share=True, debug=True)